In [1]:
import time
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Đọc đúng file và đúng sheet dữ liệu nấm
file_name = 'Mushroom_Preprocessed_Dataset_Cleaned.xlsx'
df = pd.read_excel(file_name, sheet_name=0)
print(f"-> Đã tải dữ liệu nấm: {df.shape[0]} dòng, {df.shape[1]} cột.")

# 2. Mã hóa One-Hot Encoding toàn bộ dữ liệu
df_encoded = pd.get_dummies(df)
df_encoded = df_encoded > 0

# --- NHIỆM VỤ 1: Vòng lặp 3 minsup x 3 minconf (Dùng ngưỡng an toàn từ 0.3 để không bao giờ bị tràn RAM) ---
minsup_list = [0.3, 0.4, 0.5]
minconf_list = [0.6, 0.7, 0.8]

stats_results = []

for minsup in minsup_list:
  for minconf in minconf_list:
    start_time = time.time()  # Đo thời gian thực thi chính xác bằng time.time()

    frequent_itemsets = apriori(
        df_encoded, min_support=minsup, use_colnames=True
    )
    rules = association_rules(
        frequent_itemsets, metric='confidence', min_threshold=minconf
    )

    elapsed_time = time.time() - start_time

    stats_results.append({
        'Minsup': minsup,
        'Minconf': minconf,
        'Số tập phổ biến': len(frequent_itemsets),
        'Tổng số luật': len(rules),
        'Thời gian (giây)': round(elapsed_time, 4),
    })

df_stats = pd.DataFrame(stats_results)
print('\n=== BẢNG THỐNG KÊ ĐA NGƯỠNG ===')
display(df_stats)

# --- NHIỆM VỤ 2: Lọc luật lấy mục tiêu (consequents chứa class=poisonous hoặc class=edible) ---
# Chọn bộ ngưỡng tối ưu (minsup = 0.3, minconf = 0.7) để sinh ra số lượng luật phong phú
chosen_minsup = 0.3
chosen_minconf = 0.7

frequent_itemsets_final = apriori(
    df_encoded, min_support=chosen_minsup, use_colnames=True
)
rules_final = association_rules(
    frequent_itemsets_final, metric='confidence', min_threshold=chosen_minconf
)


# Hàm lọc chính xác: kiểm tra vế phải (consequents) có chứa thuộc tính class nấm hay không
def filter_target_class(row):
  for item in row['consequents']:
    item_str = str(item)
    if 'class_' in item_str or 'class' in item_str.lower():
      return True
  return False


rules_filtered = rules_final[rules_final.apply(filter_target_class, axis=1)]
print(
    f"\n=> Đã lọc thành công {len(rules_filtered)} luật có consequents chứa"
    " mục tiêu class."
)

# --- NHIỆM VỤ 3: Xuất kết quả ra file Excel hoàn chỉnh gồm 2 Sheet ---
output_file = 'Mushroom_Rules_Optimal.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
  df_stats.to_excel(writer, sheet_name='1. Thống kê đa ngưỡng', index=False)
  rules_filtered.to_excel(
      writer, sheet_name='2. Luật mục tiêu Class', index=False
  )

print(f"\n=> Xuất file Excel thành công mỹ mãn: '{output_file}'!")

-> Đã tải dữ liệu nấm: 5 dòng, 4 cột.

=== BẢNG THỐNG KÊ ĐA NGƯỠNG ===


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Minsup,Minconf,Số tập phổ biến,Tổng số luật,Thời gian (giây)
0,0.3,0.6,1,0,0.0036
1,0.3,0.7,1,0,0.0027
2,0.3,0.8,1,0,0.0027
3,0.4,0.6,1,0,0.0025
4,0.4,0.7,1,0,0.0025
5,0.4,0.8,1,0,0.0025
6,0.5,0.6,1,0,0.0024
7,0.5,0.7,1,0,0.0026
8,0.5,0.8,1,0,0.0025



=> Đã lọc thành công 0 luật có consequents chứa mục tiêu class.

=> Xuất file Excel thành công mỹ mãn: 'Mushroom_Rules_Optimal.xlsx'!


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag